# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/pilot/` (local only, gitignored).

In [1]:
import warnings
import plotly.io as pio

from analysis.pilot.parser import load_pilot_data
from analysis.pilot.figures import (
    fig_completion_status,
    fig_trial_duration_per_subject,
    fig_moves_per_subject,
    fig_duration_progression,
    fig_moves_progression,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_demographics,
    fig_pairwise_distance_distribution,
)

pio.renderers.default = "browser"

## 0. Load data

In [2]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = load_pilot_data("data/pilot")

df_trials = data["trials"]
df_status = data["status"]

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nTrials dataframe: {df_trials.shape[0]} rows × {df_trials.shape[1]} cols")
print(df_status["completion_status"].value_counts().to_string())

[UserWarning] Participant 6150fbd25056424b64062835: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 69f5f8a6cd820e91eab3f3f7: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 698095549e3f340d0843b024: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 5d9e3ec9611b0b0017b14b9a: revoked consent (status=REJECTED), excluded from trials.
[UserWarning] Participant 69f5bb76f845c6ae7f522328: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 60001f74b9d8d70009041d15: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 67e069199f5b036960b0cc5f: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 694555b4beadda60a5901ec0: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 697cb3

## 1. Completion status

In [3]:
fig_completion_status(df_status).show()

## 2. Trial duration per subject

In [4]:
fig_trial_duration_per_subject(df_trials).show()

## 3. Number of moves per subject

In [5]:
fig_moves_per_subject(df_trials).show()

## 4. Trial duration over task progression

In [6]:
fig_duration_progression(df_trials).show()

## 5. Moves over task progression

In [7]:
fig_moves_progression(df_trials).show()

## 6. Trial duration vs. number of moves

In [8]:
fig_duration_vs_moves(df_trials).show()

## 7. Within-subject variability and reliability

In [9]:
fig_within_subject_variability(df_trials).show()

## 8. Participant demographics

In [10]:
fig_demographics(df_trials).show()

## 9. Pairwise distance distribution

In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

from analysis.pilot.simulate_null_distances import simulate as _sim_null

# Shared null: same K as images_per_trial, enough trials for a stable reference
null_distances = _sim_null(num_dots=20, num_trials=1000, seed=42)
print(f"Null: {len(null_distances):,} distances  mean={null_distances.mean():.3f}  sd={null_distances.std():.3f}")

fig_pairwise_distance_distribution(df_trials, null_distribution=null_distances).show()

Null: 190,000 distances  mean=0.369  sd=0.175


## Summary statistics and between-cohort comparisons

In [12]:
summary = (
    df_trials
    .assign(
        rt_s=df_trials["rt"] / 1000,
        qc_flag_int=df_trials["qc_flag"].astype(int),
    )
    .groupby("task_version")
    .agg(
        n_subjects=("participant_id",  "nunique"),
        n_trials=("trial_number",      "count"),
        rt_s_mean=("rt_s",             "mean"),
        rt_s_median=("rt_s",           "median"),
        rt_s_sd=("rt_s",               "std"),
        rt_s_min=("rt_s",              "min"),
        rt_s_max=("rt_s",              "max"),
        moves_mean=("n_moves",         "mean"),
        moves_median=("n_moves",       "median"),
        moves_sd=("n_moves",           "std"),
        qc_flag_rate=("qc_flag_int",   "mean"),
    )
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary

task_version,v1,v2
n_subjects,15.00,9.00
n_trials,150.00,90.00
rt_s_mean,111.19,70.21
rt_s_median,90.15,64.56
rt_s_sd,93.77,19.01
rt_s_min,22.26,60.59
rt_s_max,926.99,227.23
moves_mean,36.11,27.86
moves_median,26.00,27.00
moves_sd,24.02,8.20


### Between-cohort statistical comparisons

Mann-Whitney U on per-subject aggregates (one value per subject — no pseudo-replication).  
Effect size: rank-biserial correlation *r* = 1 − 2U / (n₁ × n₂), where |r| ≥ 0.5 is conventionally large.

In [13]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# ---------------------------------------------------------------------------
# Per-subject aggregates
# ---------------------------------------------------------------------------

subj_rt = (
    df_trials.assign(rt_s=df_trials["rt"] / 1000)
    .groupby(["participant_id", "task_version"])["rt_s"]
    .mean()
    .reset_index()
    .rename(columns={"rt_s": "mean_rt_s"})
)

subj_moves = (
    df_trials
    .groupby(["participant_id", "task_version"])["n_moves"]
    .mean()
    .reset_index()
    .rename(columns={"n_moves": "mean_moves"})
)


def _parse_pairwise(pw_json):
    if pd.isna(pw_json) or pw_json == "":
        return {}
    try:
        items = json.loads(pw_json)
    except (json.JSONDecodeError, TypeError):
        return {}
    return {tuple(sorted([it["src1"], it["src2"]])): it["distance"] for it in items}


def _subject_snr(df_s):
    pair_obs = defaultdict(list)
    for pw_json in df_s["pairwise_distances"]:
        for pair, dist in _parse_pairwise(pw_json).items():
            pair_obs[pair].append(dist)
    d1, d2 = [], []
    for obs in pair_obs.values():
        if len(obs) >= 2:
            d1.append(obs[0])
            d2.append(obs[1])
    if not d1:
        return np.nan
    all_dists = [d for pw_json in df_s["pairwise_distances"]
                 for d in _parse_pairwise(pw_json).values()]
    sigma_d = float(np.std(all_dists)) if len(all_dists) > 1 else 1.0
    mean_abs_diff = float(np.mean(np.abs(np.array(d1) - np.array(d2))))
    return sigma_d / mean_abs_diff if mean_abs_diff > 0 else np.nan


subj_snr = (
    df_trials
    .groupby(["participant_id", "task_version"])
    .apply(_subject_snr, include_groups=False)
    .reset_index()
    .rename(columns={0: "snr"})
)

# ---------------------------------------------------------------------------
# Mann-Whitney U helper
# ---------------------------------------------------------------------------

def mwu_compare(df, value_col, group_col="task_version"):
    groups = sorted(df[group_col].unique())
    assert len(groups) == 2, "expected exactly two groups"
    # Cast to float — some columns (e.g. n_moves) land as object dtype
    a = df.loc[df[group_col] == groups[0], value_col].dropna().astype(float).values
    b = df.loc[df[group_col] == groups[1], value_col].dropna().astype(float).values
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * U / (len(a) * len(b))
    return pd.Series({
        f"n (v{groups[0]:g})":      len(a),
        f"n (v{groups[1]:g})":      len(b),
        f"median (v{groups[0]:g})": round(float(np.median(a)), 2),
        f"median (v{groups[1]:g})": round(float(np.median(b)), 2),
        "U":                        round(U, 1),
        "p (two-sided)":            round(p, 4),
        "r (rank-biserial)":        round(r, 3),
    })


results = pd.DataFrame({
    "Trial duration (s)":  mwu_compare(subj_rt,    "mean_rt_s"),
    "Moves per trial":     mwu_compare(subj_moves,  "mean_moves"),
    "SNR (σ_d/mean|Δd|)": mwu_compare(subj_snr,    "snr"),
}).T

results

,n (v1),n (v2),median (v1),median (v2),U,p (two-sided),r (rank-biserial)
Trial duration (s),15.0,9.0,102.27,66.77,110.0,0.0123,-0.630
Moves per trial,15.0,9.0,27.40,25.90,71.0,0.8580,-0.052
SNR (σ_d/mean|Δd|),15.0,9.0,1.06,1.09,73.0,0.7656,-0.081


### Distance distribution vs null

Per-subject KS statistic D against the shared null (random placement).  
D ∈ [0, 1]; higher = further from the null = more structured arrangement.  
Then Mann-Whitney U tests whether one cohort is systematically further from null than the other.

In [14]:
from scipy.stats import ks_2samp

# null_distances defined in cell 9 above (shared across cohorts)

def _subject_ks_vs_null(df_s, null):
    dists = [d for pw_json in df_s["pairwise_distances"]
             for d in _parse_pairwise(pw_json).values()]
    D, _ = ks_2samp(dists, null)
    return D

subj_ks = (
    df_trials
    .groupby(["participant_id", "task_version"])
    .apply(_subject_ks_vs_null, null=null_distances, include_groups=False)
    .reset_index()
    .rename(columns={0: "ks_vs_null"})
)

print("KS statistic vs null (per subject):")
print(subj_ks.groupby("task_version")["ks_vs_null"].describe().round(3))
print()

ks_result = mwu_compare(subj_ks, "ks_vs_null")
pd.DataFrame({"KS distance from null": ks_result})

KS statistic vs null (per subject):
              count   mean    std    min    25%    50%    75%    max
task_version                                                        
1.0            15.0  0.114  0.057  0.049  0.083  0.101  0.125  0.289
2.0             9.0  0.148  0.050  0.089  0.124  0.146  0.152  0.267



,KS distance from null
n (v1),15.0000
n (v2),9.0000
median (v1),0.1000
median (v2),0.1500
U,30.0000
p (two-sided),0.0274
r (rank-biserial),0.5560


## 10. Temporal engagement

Do subjects work throughout the trial, or front-load moves and sit idle?

- **Left**: cumulative move fraction vs. time fraction within trial. Diagonal = uniform activity; curve bending top-left = front-loading.
- **Right**: average move rate (moves/s, 5 s bins) over absolute time. Dashed line = 60 s (V2 timer unlock).

In [ ]:
from analysis.pilot.figures import fig_move_temporal_profile

fig_move_temporal_profile(df_trials).show()

### Idle tail fraction

**Idle tail fraction** = (RT - t_last_move) / RT per trial: the proportion of trial time spent doing nothing after the last move.
Aggregated per subject (mean across trials), then compared with Mann-Whitney U.

In [ ]:
def _idle_tail_fraction(row):
    """Fraction of trial time after the last move: (RT - t_last) / RT."""
    try:
        moves = json.loads(row["moves"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    ts = [m["t"] for m in moves if isinstance(m.get("t"), (int, float))]
    if not ts or pd.isna(row["rt"]) or row["rt"] <= 0:
        return np.nan
    return (row["rt"] - max(ts)) / row["rt"]


subj_idle = (
    df_trials
    .assign(idle_tail=df_trials.apply(_idle_tail_fraction, axis=1))
    .groupby(["participant_id", "task_version"])["idle_tail"]
    .mean()
    .reset_index()
)

print("Idle tail fraction per cohort:")
print(subj_idle.groupby("task_version")["idle_tail"].describe().round(3))
print()

pd.DataFrame({"Idle tail fraction": mwu_compare(subj_idle, "idle_tail")})